# Recursive Club Split Visualization

This notebook loads saved outputs for runs A, B, and C and visualizes their cluster trees interactively.

In [1]:
from pathlib import Path
import pandas as pd
import json

# Select which tumor to visualize
tumor = "t24"  # one of: t24, t25, t31, t46, t57

def find_latest_results_bundle(tumor):
    cwd = Path.cwd().resolve()
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

    base = project_root / "data" / tumor / "results_bundle"
    if not base.exists():
        raise FileNotFoundError(
            f"No results_bundle folder for {tumor}. Run scripts/run_club_split.py --tumor {tumor} first."
        )

    run_dirs = sorted([p for p in base.glob("triple_run_*") if p.is_dir()], key=lambda p: p.name)
    if not run_dirs:
        raise FileNotFoundError(f"No triple_run_* folders found in {base}")

    return run_dirs[-1]

bundle_dir = find_latest_results_bundle(tumor)
print("Tumor:", tumor)
print("Using results bundle:", bundle_dir)

Tumor: t24
Using results bundle: /data/safe/ranade/pythoncode/Multinomial_EM/data/t24/results_bundle/triple_run_20261006_120619


In [2]:
def read_table(bundle_dir, tumor, stem):
    parquet_path = bundle_dir / f"{tumor}_{stem}.parquet"
    csv_path = bundle_dir / f"{tumor}_{stem}.csv"

    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)

    raise FileNotFoundError(f"Could not find {tumor}_{stem} as parquet or csv in {bundle_dir}")

runs = {}
for label in ["A", "B", "C"]:
    runs[label] = {
        "tree": read_table(bundle_dir, tumor, f"{label}_cluster_tree"),
        "leaf": read_table(bundle_dir, tumor, f"{label}_leaf_clusters"),
        "assign": read_table(bundle_dir, tumor, f"{label}_cell_assignments"),
        "unassigned": read_table(bundle_dir, tumor, f"{label}_unassigned_cells"),
    }

manifest_path = bundle_dir / f"{tumor}_run_manifest.json"
manifest = None
if manifest_path.exists():
    with open(manifest_path, "r") as f:
        manifest = json.load(f)

print("Loaded runs:", list(runs.keys()))
if manifest is not None:
    print("Manifest timestamp:", manifest.get("saved_at", "unknown"))
    print("Run params:", manifest.get("params", {}))

FileNotFoundError: Could not find t24_A_cluster_tree as parquet or csv in /data/safe/ranade/pythoncode/Multinomial_EM/data/t24/results_bundle/triple_run_20261006_120619

In [ ]:
summary_rows = []
for label, data_dict in runs.items():
    summary_rows.append({
        "run": label,
        "n_tree_nodes": len(data_dict["tree"]),
        "n_leaf_clusters": len(data_dict["leaf"]),
        "n_assigned_cells": data_dict["assign"]["cell_id"].nunique() if len(data_dict["assign"]) else 0,
        "n_unassigned_cells": data_dict["unassigned"]["cell_id"].nunique() if len(data_dict["unassigned"]) else 0,
    })

summary_df = pd.DataFrame(summary_rows).sort_values("run").reset_index(drop=True)
summary_df

In [ ]:
import plotly.express as px

def plot_cluster_tree(label="C", figure_type="icicle"):
    if label not in runs:
        raise ValueError(f"Unknown run label: {label}. Choose from {list(runs.keys())}.")

    tree_df = runs[label]["tree"].copy()

    # Normalize expected schema in case csv round-trips change dtype.
    for col in ["id", "parent"]:
        tree_df[col] = tree_df[col].astype(str)
    tree_df["value"] = tree_df["value"].astype(float)

    fig_type = figure_type.lower()
    if fig_type == "icicle":
        fig = px.icicle(
            tree_df,
            names="id",
            parents="parent",
            values="value",
            color="depth",
            color_continuous_scale="Blues",
        )
    elif fig_type == "sunburst":
        fig = px.sunburst(
            tree_df,
            names="id",
            parents="parent",
            values="value",
            color="depth",
            color_continuous_scale="Blues",
        )
    elif fig_type == "treemap":
        fig = px.treemap(
            tree_df,
            names="id",
            parents="parent",
            values="value",
            color="depth",
            color_continuous_scale="Blues",
        )
    else:
        raise ValueError("figure_type must be one of: icicle, sunburst, treemap")

    fig.update_traces(
        texttemplate="%{label}<br>%{value:.0f}",
        hovertemplate=(
            "<b>%{label}</b><br>"
            "Cells: %{value:.0f}<br>"
            "Parent: %{parent}<br>"
            "Depth: %{color}<extra></extra>"
        ),
    )
    fig.update_layout(
        title=f"{tumor} - Run {label}: Recursive Club Tree ({fig_type})",
        margin=dict(t=60, l=20, r=20, b=20),
    )
    fig.show()

plot_cluster_tree(label="C", figure_type="icicle")

In [ ]:
# Quick comparison: leaf cluster sizes across A/B/C
compare_rows = []
for label, data_dict in runs.items():
    tmp = data_dict["leaf"][["cluster_label", "n_cells"]].copy()
    tmp["run"] = label
    compare_rows.append(tmp)

leaf_compare_df = pd.concat(compare_rows, ignore_index=True)
leaf_compare_df.sort_values(["run", "n_cells"], ascending=[True, False]).head(30)

In [ ]:
# Inspect the full tree table for one run (internal + leaf nodes)
inspect_label = "C"  # "A", "B", or "C"

tree_view = runs[inspect_label]["tree"].sort_values(["depth", "id"]).reset_index(drop=True)
print(f"{tumor} run {inspect_label}: {len(tree_view)} nodes "
      f"({tree_view['is_leaf'].sum()} leaves)")
tree_view

In [ ]:
# Cell-level assignments: every cell and its leaf cluster (unassigned listed separately)
assign_view = runs[inspect_label]["assign"]
unassigned_view = runs[inspect_label]["unassigned"]

print(f"Assigned cells: {len(assign_view)}")
print(f"Unassigned cells: {len(unassigned_view)}")

# Preview assignments grouped by cluster
assign_view.groupby("leaf_cluster").size().sort_values(ascending=False).to_frame("n_cells")

In [ ]:
# Leaf clusters table: one row per final cluster
leaf_view = runs[inspect_label]["leaf"].sort_values("n_cells", ascending=False).reset_index(drop=True)
leaf_view

In [ ]:
# Reconstruct the original dictionaries from the saved tables
# final_clusters: {cluster_label: [cell, cell, ...]}
# unassigned:     {node_label: [cell, cell, ...]}  (cells dropped at that node)

final_clusters_dict = (
    runs[inspect_label]["assign"]
    .groupby("leaf_cluster")["cell_id"]
    .apply(list)
    .to_dict()
)

unassigned_dict = (
    runs[inspect_label]["unassigned"]
    .groupby("node")["cell_id"]
    .apply(list)
    .to_dict()
)

print(f"{tumor} run {inspect_label}: {len(final_clusters_dict)} final clusters, "
      f"{len(unassigned_dict)} nodes with unassigned cells")

# Peek at sizes
print("\nCluster sizes (first 15):")
for k in sorted(final_clusters_dict, key=lambda k: (-len(final_clusters_dict[k]), k))[:15]:
    print(f"  {k}: {len(final_clusters_dict[k])} cells")

print("\nUnassigned by node:")
for k in sorted(unassigned_dict):
    print(f"  {k}: {len(unassigned_dict[k])} cells")

In [ ]:
# Look at the actual cell lists for a specific cluster or unassigned node
cluster_to_view = "A.0"   # change to any key, e.g. "A.0.1" or a node from the unassigned list

if cluster_to_view in final_clusters_dict:
    cells_here = final_clusters_dict[cluster_to_view]
    print(f"{cluster_to_view}: {len(cells_here)} cells (final cluster)")
elif cluster_to_view in unassigned_dict:
    cells_here = unassigned_dict[cluster_to_view]
    print(f"{cluster_to_view}: {len(cells_here)} cells (unassigned at this node)")
else:
    cells_here = []
    print(f"'{cluster_to_view}' not found. Available final clusters:\n",
          sorted(final_clusters_dict.keys())[:20], "...")

cells_here

In [ ]:
plot_cluster_tree(label="A", figure_type="icicle")
plot_cluster_tree(label="B", figure_type="icicle")
plot_cluster_tree(label="C", figure_type="icicle")

In [ ]:
stop

In [ ]:
# Load all saved tables into explicit DataFrames (these ARE the same tables
# the save cell built: tree_df, leaf_df, assign_df, unassigned_df per run)

tables = {}   # tables["A"]["tree_df"], tables["A"]["leaf_df"], ...

for label in ["A", "B", "C"]:
    tables[label] = {
        "tree_df": runs[label]["tree"],
        "leaf_df": runs[label]["leaf"],
        "assign_df": runs[label]["assign"],
        "unassigned_df": runs[label]["unassigned"],
    }

# Quick integrity check: assignments + unassigned should partition all input cells
for label in ["A", "B", "C"]:
    n_assigned = tables[label]["assign_df"]["cell_id"].nunique()
    n_unassigned = tables[label]["unassigned_df"]["cell_id"].nunique()
    print(f"Run {label}: {n_assigned} assigned + {n_unassigned} unassigned "
          f"= {n_assigned + n_unassigned} total cells")

# Example: all four tables for run C
tables["C"]["tree_df"].head(), tables["C"]["leaf_df"].head(), \
    tables["C"]["assign_df"].head(), tables["C"]["unassigned_df"].head()

In [ ]:
len(runs["A"]["assign"]["leaf_cluster"].unique())

In [ ]:
runs["A"]["unassigned"].head()

In [ ]:
# Export assignment + unassigned tables for ALL tumors and runs into one
# shareable folder inside the project.

import shutil
from datetime import datetime

tumors_all = ["t24", "t25", "t31", "t46", "t57"]
run_labels = ["A", "B", "C"]

# Same root logic as the loading cell
cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
print("Project root:", project_root)

staging = project_root / "club_assignments_for_sharing"
staging.mkdir(exist_ok=True)

copied = []
missing = []
for t in tumors_all:
    try:
        bdir = find_latest_results_bundle(t)
    except FileNotFoundError:
        missing.append(t)
        continue
    for label in run_labels:
        for stem in ["cell_assignments", "unassigned_cells"]:
            src = bdir / f"{t}_{label}_{stem}.csv"
            if src.exists():
                shutil.copy2(src, staging / src.name)
                copied.append(src.name)
            else:
                missing.append(src.name)

print(f"Copied {len(copied)} files to {staging}")
if missing:
    print("Missing:", missing)